# 02 — Index and ingestion

This notebook uses the application modules. It does not define a second embedding model or index mapping.

The app defaults are:

- model: `sentence-transformers/all-mpnet-base-v2`
- dimension: 768
- distance: cosine similarity, with L2-normalized vectors
- chunks: 300 words, overlap 100
- index: `documents`


In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.constants import (
    EMBEDDING_DIMENSION,
    EMBEDDING_MODEL_PATH,
    OPENSEARCH_INDEX,
    TEXT_CHUNK_OVERLAP,
    TEXT_CHUNK_SIZE,
)
from src.embeddings import generate_embeddings
from src.extract import extract_text
from src.ingestion import bulk_index_documents, create_index
from src.opensearch import get_opensearch_client
from src.utils import chunk_text

print(EMBEDDING_MODEL_PATH, EMBEDDING_DIMENSION, TEXT_CHUNK_SIZE, TEXT_CHUNK_OVERLAP, OPENSEARCH_INDEX)


Set `PDF_PATH` to a local PDF, DOCX, TXT, or Markdown file. The next cell indexes it with the same functions the Upload page uses. OpenSearch must be running on `localhost:9200`.


In [ ]:
PDF_PATH = ""  # for example: "../uploaded_files/example.pdf"

if not PDF_PATH:
    print("Set PDF_PATH to index a file. Showing the call sequence only.")
else:
    client = get_opensearch_client()
    create_index(client)
    text = extract_text(PDF_PATH)
    chunks = chunk_text(text, chunk_size=TEXT_CHUNK_SIZE, overlap=TEXT_CHUNK_OVERLAP)
    vectors = generate_embeddings(chunks)
    name = Path(PDF_PATH).name
    docs = [
        {
            "doc_id": f"{name}_{i}",
            "text": chunk,
            "embedding": vector,
            "document_name": name,
        }
        for i, (chunk, vector) in enumerate(zip(chunks, vectors))
    ]
    success, errors = bulk_index_documents(docs)
    print({"chunks": len(chunks), "indexed": success, "errors": len(errors)})
